In [7]:
!pip install -q pyannote.audio pyannote.metrics

import os, shutil, pathlib
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 16.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!ls /kaggle/input/          # confirm both slugs

datasets


In [2]:
import pathlib, shutil, os

ROOT = pathlib.Path("/kaggle/input")

# Find the directory that actually contains the scripts, wherever it landed.
CODE = next(p.parent for p in ROOT.rglob("stage3_diarize.py"))
# Find the directory that actually contains the WAVs.
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
REF = next(p.parent for p in ROOT.rglob("clip_meta.csv"))
WORK = "/kaggle/working/data"

print("CODE  =", CODE)
print("AUDIO =", AUDIO, f"({len(list(AUDIO.glob('*.wav')))} wavs)")
print("REF   =", REF, f"({len(list(REF.rglob('*')))} files)")

CODE  = /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code
AUDIO = /kaggle/input/datasets/ritankarmondal/sarvam-diar-audio/wav (99 wavs)
REF   = /kaggle/input/datasets/ritankarmondal/sarvam-diar-code/upload_code/ref (205 files)


In [3]:
pathlib.Path(WORK).mkdir(parents=True, exist_ok=True)
shutil.copytree(REF, f"{WORK}/ref", dirs_exist_ok=True)
for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")
mf = next(ROOT.rglob("manifest.jsonl"), None)
if mf: shutil.copy(mf, WORK)

print("ref files:", len(list(pathlib.Path(f'{WORK}/ref').rglob('*'))))
print("scripts  :", [p.name for p in pathlib.Path('/kaggle/working').glob('*.py')])

ref files: 205
scripts  : ['stage1_extract.py', 'stage3_diarize.py', 'stage3_score.py', 'stage2_parse_refs.py', 'build_notebooks.py']


In [11]:
import shutil, pathlib
PREV = pathlib.Path("/kaggle/input/notebooks/ritankarmondal/sarvam-initial")   # fix the slug
DST  = pathlib.Path("/kaggle/working/data")

assert (PREV / "data").exists(), sorted(p.name for p in PREV.iterdir())
DST.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV / "data", DST, dirs_exist_ok=True)

print("ref  :", len(list((DST / "ref/rttm").glob("*.rttm"))))
print("hyp  :", len(list((DST / "hyp").rglob("*.rttm"))))

ref  : 100
hyp  : 99


In [12]:
!find /kaggle/working/data/hyp/pyannote31/rttm -name "*.rttm" | wc -l

99


In [ ]:
!python stage3_score.py --data data --systems pyannote31 --diagnostic